# B. 독립변수 16개 추출 버전(t_test + logistic regression)


## 1. 라이브러리 설치

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. 데이터 불러오기 및 탐색 함수


In [ ]:
import pandas as pd

path = '/content/drive/MyDrive/동국대학원/연구 과제/양자컴퓨팅 기반 머신러닝 하이브리드 모델을 활용한 기업신용평가모형/data/FILE2_중공업.xls'
df = pd.read_excel(path)
df.tail(5)

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd


# ============================================================
# 1. 설정
# ============================================================


# 결과 파일 저장 여부
SAVE_RESULTS = False
OUTPUT_DIR = "eda_results"

# 고상관 변수 기준
CORR_THRESHOLD = 0.95

ID = "회사코드"
INDUSTRY = "산업코드"
TARGET = "정상/폐업"
DATE = "재무비율기준/결산기준일"
AUDIT = "감사의견코드"

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")


# ============================================================
# 2. PDF 참고: 산업코드 사전
# ============================================================

INDUSTRIES = {
    "D21": "펄프, 종이 및 종이제품 제조업",
    "D23": "코크스, 석유정제품 및 핵연료 제조업",
    "D24": "화합물 및 화학제품 제조업",
    "D25": "고무 및 플라스틱제품 제조업",
    "D26": "비금속 광물제품 제조업",
    "D27": "제1차 금속산업",
    "D28": "조립금속제품 제조업: 기계 및 가구 제외",
    "D29": "기타 기계 및 장비 제조업",
    "D30": "컴퓨터 및 사무용 기기제조업",
    "D31": "기타 전기기계 및 전기변환장치 제조업",
    "D32": "전자부품, 영상, 음향 및 통신장비 제조업",
    "D33": "의료, 정밀, 공학기기 및 시계 제조업",
    "D34": "자동차 및 트레일러 제조업",
    "D35": "기타 운송장비 제조업",
}


# ============================================================
# 3. PDF 참고: 변수 분류
#
# 변수명으로 직접 매칭한다.
# 비율, 금액, 배수, 기간 등이 섞여 있으므로
# 모든 수치 변수를 같은 단위라고 가정하면 안 된다.
# ============================================================

GROUP_NAMES = {
    "성장성": """
        매출액증가율|영업이익증가율|순이익증가율|총자산증가율|
        자기자본증가율|법인세차감전순이익증가율|유동자산증가율|
        유형자산증가율|재고자산증가율|종업원수증가율
    """,

    "수익성": """
        매출원가율|판관비율|영업이익률|법인세차감전순이익률|
        당기순이익율|ROIC|ROE|EBITDA마진율|재료비/매출액|
        노무비/매출액|경비/매출액|총자산법인세차감전순이익률|
        순금융비용부담율|자본금법인세차감전순이익률|감가상각률|
        감가상각비대총비용비율|법인세차감전순이익이자보상율|
        금융비용대총부채비율|금융비용대총비용비율|매출총이익률|
        배당성향|배당율|사내유보대자기자본비율|사내유보율|
        순외환손익대매출액|연구개발비대매출액|인건비대총비용비율|
        자기자본법인세차감전순이익률|자기자본배당률|자본금순이익률|
        조세공과대총비용비율|조세대조세차감전순이익률|차입금평균이자율|
        총자산순이익률(ROA)|총자산영업이익률|EBITDA(천원)|
        영업외수지율|경영자본영업이익율|수지비율|
        현금영업이익/매출액|현금흐름/매출액|FCF/매출액
    """,

    "안정성": """
        이자보상배수(배)|EBITDA/금융비용(배)|EBITDA/총차입금|
        차입금의존도|단기성차입금/총차입금|고정장기적합율|
        금융비용부담율|부채비율|자기자본비율|차입금/매출액|
        유동비율|EBITDA/단기차입금|차입금/자기자본비율|
        장단기대여금/총자산|적립금비율|비유동부채대순운전자본비율|
        비유동부채비율|비유동비율|당좌비율|매입채무대재고자산비율|
        매출채권대매입채무비율|매출채권대상/제품비율|법인세부담률|
        순운전자본대총자본비율|유동부채비율|유동자산구성비율|
        유보액대납입자본배율|유보액대총자산비율|
        재고자산대순운전자본비율|재고자산대유동자산비율|
        재무레버리지|차입금대월매출액|총현금흐름대당기상환부채|
        총현금흐름대매출액비율|총현금흐름대부채비율|
        총현금흐름대유동부채|총현금흐름대차입금비율|
        총현금흐름대총자본비율|비유동자산비율|단기부채상환능력|
        부채상환계수|총자산부채비율|적립금총자산비율|
        적립금비율2|현금비율|현금영업이익/금융비용|
        현금영업이익/단기차입금|현금영업이익/총차입금|
        현금흐름/총차입금|영업활동조달현금/금융비용|
        영업활동조달현금/차입금|FCF/차입금
    """,

    "활동성": """
        매출채권회전율(회)|재고자산회전율(회)|순운전자본회전율(회)|
        총자본회전율(회)|회전기간|비유동자산회전율(회)|
        매입채무회전기간|매입채무회전율(회)|매출채권회전기간|
        상(제)품회전율(회)|순영업자본회전율(회)|영업자본회전율(회)|
        순운전자본회전기간|원부재료회전율(회)|유동자산회전율(회)|
        유형자산회전율(회)|자기자본회전율(회)|자본금회전율(회)|
        재고자산회전기간|재공품회전율(회)|투하자산회전율(회)
    """,

    "생산성": """
        기계투자효율|노동소득분배율|부가가치(천원)|부가가치율|
        설비투자효율|이윤분배율|종업원1인당법인세차감전순이익|
        종업원1인당기계장비율|종업원1인당노동장비율|
        종업원1인당매출액|종업원1인당부가가치|종업원1인당순이익|
        종업원1인당영업이익|종업원1인당인건비|
        종업원1인당자본집약도|총자본투자효율|자본분배율
    """,

    "비재무/규모": """
        업력|경영자본|부가가치|종업원수|감사의견코드
    """,
}

GROUPS = {
    variable.strip(): group
    for group, variables in GROUP_NAMES.items()
    for variable in variables.split("|")
    if variable.strip()
}


# ============================================================
# 4. 데이터 읽기
# ============================================================

def read_data(path):
    """
    실제 파일 형식에 따라 xls/xlsx 엔진을 선택한다.

    회사코드는 식별자이므로 문자열로 읽는다.
    원본 파일 자체는 수정하지 않는다.
    """

    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"파일이 없습니다: {path.resolve()}")

    with path.open("rb") as f:
        signature = f.read(4)

    engine = "openpyxl" if signature == b"PK\x03\x04" else "xlrd"

    df = pd.read_excel(
        path,
        sheet_name=0,
        engine=engine,
        dtype={
            ID: "string",
            INDUSTRY: "string",
            AUDIT: "string",
        },
    )

    df.columns = df.columns.astype(str).str.strip()

    if df.columns.duplicated().any():
        raise ValueError("중복된 열 이름이 있습니다.")

    required = {ID, INDUSTRY, TARGET, DATE, AUDIT}
    missing_columns = required - set(df.columns)

    if missing_columns:
        raise ValueError(f"필수 컬럼이 없습니다: {missing_columns}")

    # 문자열의 앞뒤 공백만 정리
    for col in [ID, INDUSTRY, AUDIT]:
        df[col] = df[col].str.strip().replace("", pd.NA)

    return df


# ============================================================
# 5. 전체 EDA
# ============================================================

def explore_data(df, corr_threshold=0.95):
    """
    원자료를 임의 대치하거나 이상값을 삭제하지 않는 EDA.

    반환값
    -------
    x       : 수치형 분석용 복사본
    dates   : 파싱한 결산일
    tables  : 탐색 결과 DataFrame 사전
    summary : 전체 요약 사전
    """

    if not 0 <= corr_threshold <= 1:
        raise ValueError("상관계수 기준은 0~1 사이여야 합니다.")

    excluded = {ID, INDUSTRY, TARGET, DATE, AUDIT}

    numeric_features = [
        col for col in df.columns
        if col not in excluded
    ]

    # 숫자 변환 실패를 별도로 집계
    converted = df[numeric_features].apply(
        pd.to_numeric,
        errors="coerce",
    )

    conversion_failures = (
        df[numeric_features].notna()
        & converted.isna()
    )

    infinite = pd.DataFrame(
        np.isinf(converted),
        index=df.index,
        columns=numeric_features,
    )

    # 분석 복사본에서만 무한대를 NaN으로 처리
    # 원본의 결측, 0, 음수, 큰 수는 임의 변경하지 않음
    x = converted.replace([np.inf, -np.inf], np.nan)

    # --------------------------------------------------------
    # 결산일
    # --------------------------------------------------------

    date_text = (
        df[DATE]
        .astype("string")
        .str.replace(r"\.0$", "", regex=True)
        .str.strip()
    )

    dates = pd.to_datetime(
        date_text,
        format="%Y%m%d",
        errors="coerce",
    )

    year = dates.dt.year.astype("Int64").rename("결산연도")

    # 타깃 결측이 있더라도 교차표에서 제외되지 않게 표시
    labels = df[TARGET].astype("string").fillna("(결측)")
    industries = df[INDUSTRY].fillna("(결측)")
    audits = df[AUDIT].fillna("(결측)")

    # --------------------------------------------------------
    # A. 변수 프로파일
    # --------------------------------------------------------

    profile = pd.DataFrame({
        "변수": df.columns,
        "자료형": df.dtypes.astype(str).values,
        "분류": [
            GROUPS.get(col, "식별/분류/기준일 또는 미분류")
            for col in df.columns
        ],
        "결측수": df.isna().sum().values,
        "결측비율": df.isna().mean().values,
        "고유값수": df.nunique(dropna=True).values,
    })

    # --------------------------------------------------------
    # B. 수치형 기술통계
    # --------------------------------------------------------

    stats = x.describe(
        percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
    ).T

    stats.index.name = "변수"

    valid_count = x.notna().sum().replace(0, np.nan)

    stats["결측비율_분석용"] = x.isna().mean()
    stats["0개수"] = x.eq(0).sum()
    stats["0비율_전체행기준"] = x.eq(0).mean()
    stats["0비율_유효값기준"] = x.eq(0).sum() / valid_count
    stats["음수개수"] = x.lt(0).sum()
    stats["음수비율_유효값기준"] = x.lt(0).sum() / valid_count
    stats["무한대개수"] = infinite.sum()
    stats["숫자변환실패수"] = conversion_failures.sum()
    stats["왜도"] = x.skew()
    stats["첨도"] = x.kurt()

    # --------------------------------------------------------
    # C. IQR 기반 극단값 탐색
    # --------------------------------------------------------

    q1 = x.quantile(0.25)
    q3 = x.quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    outlier_flags = (
        x.lt(lower)
        | x.gt(upper)
    ) & x.notna()

    stats["IQR하한"] = lower
    stats["IQR상한"] = upper
    stats["IQR극단값수"] = outlier_flags.sum()
    stats["IQR극단값비율"] = outlier_flags.sum() / valid_count
    stats["IQR이0인지"] = iqr.eq(0)

    # 주의:
    # IQR 기준을 벗어났다는 이유만으로 오류라고 판단하지 않음.
    # 분모가 작은 재무비율은 정상적인 값도 매우 커질 수 있음.
    # IQR=0인 변수에서는 특히 기계적인 해석을 피할 것.

    # --------------------------------------------------------
    # D. 완전히 동일한 변수
    # --------------------------------------------------------

    identical_pairs = []

    for i, col1 in enumerate(numeric_features):
        for col2 in numeric_features[i + 1:]:
            if df[col1].equals(df[col2]):
                identical_pairs.append((col1, col2))

    identical_table = pd.DataFrame(
        identical_pairs,
        columns=["변수1", "변수2"],
    )

    # --------------------------------------------------------
    # E. 상태 코드 빈도
    # --------------------------------------------------------

    target_counts = (
        labels
        .value_counts(dropna=False)
        .rename_axis(TARGET)
        .reset_index(name="행수")
    )

    target_counts["비율"] = target_counts["행수"] / len(df)

    # --------------------------------------------------------
    # F. 산업/연도/감사의견과 상태 코드
    # --------------------------------------------------------

    industry_target = pd.crosstab(
        industries,
        labels,
    )

    industry_target.index.name = INDUSTRY
    industry_target.columns.name = TARGET

    industry_target.insert(
        0,
        "산업명",
        [
            INDUSTRIES.get(code, "미확인")
            for code in industry_target.index
        ],
    )

    # -1은 결산일 파싱 실패 또는 결측
    year_target = pd.crosstab(
        year.fillna(-1),
        labels,
    )

    year_target.index.name = "결산연도"

    # 각 상태 코드 내부에서 연도가 차지하는 비율
    year_share_within_target = year_target.div(
        year_target.sum(axis=0).replace(0, np.nan),
        axis=1,
    )

    # 각 연도 내부에서 상태 코드가 차지하는 비율
    target_share_within_year = year_target.div(
        year_target.sum(axis=1).replace(0, np.nan),
        axis=0,
    )

    audit_target = pd.crosstab(
        audits,
        labels,
    )

    audit_target.index.name = AUDIT

    # --------------------------------------------------------
    # G. 상태 코드별 수치 통계와 결측률
    # --------------------------------------------------------

    grouped_stats = []
    grouped_missing = []

    for label in labels.unique():
        mask = labels.eq(label)
        block = x.loc[mask]

        part = (
            block
            .agg(["count", "mean", "median", "std"])
            .T
            .rename_axis("변수")
            .reset_index()
        )

        part.insert(0, TARGET, label)
        grouped_stats.append(part)

        missing_part = (
            block.isna()
            .mean()
            .rename_axis("변수")
            .reset_index(name="결측비율")
        )

        missing_part.insert(0, TARGET, label)
        grouped_missing.append(missing_part)

    stats_by_target = pd.concat(
        grouped_stats,
        ignore_index=True,
    )

    missing_by_target = pd.concat(
        grouped_missing,
        ignore_index=True,
    )

    # --------------------------------------------------------
    # H. 상관관계
    # --------------------------------------------------------

    correlation_tables = {}

    for method in ["pearson", "spearman"]:
        corr = x.corr(
            method=method,
            min_periods=30,
        )

        pairs = []

        for i, col1 in enumerate(numeric_features):
            for col2 in numeric_features[i + 1:]:
                value = corr.loc[col1, col2]

                if pd.notna(value) and abs(value) >= corr_threshold:
                    n_pairs = (
                        x[col1].notna()
                        & x[col2].notna()
                    ).sum()

                    pairs.append({
                        "변수1": col1,
                        "변수2": col2,
                        "상관계수": value,
                        "유효쌍수": int(n_pairs),
                    })

        high_corr = pd.DataFrame(
            pairs,
            columns=["변수1", "변수2", "상관계수", "유효쌍수"],
        )

        if not high_corr.empty:
            high_corr = high_corr.sort_values(
                "상관계수",
                key=lambda s: s.abs(),
                ascending=False,
            )

        correlation_tables[f"{method}_matrix"] = corr
        correlation_tables[f"{method}_highcorr"] = high_corr

    # --------------------------------------------------------
    # I. 기업별 행 수
    # --------------------------------------------------------

    company_counts = (
        df[ID]
        .value_counts(dropna=False)
        .rename_axis(ID)
        .reset_index(name="행수")
    )

    # --------------------------------------------------------
    # J. 산업 × 연도 × 상태
    # --------------------------------------------------------

    cross = pd.DataFrame({
        "결산연도": year,
        INDUSTRY: industries,
        TARGET: labels,
    })

    year_industry_target = (
        cross
        .groupby(
            ["결산연도", INDUSTRY, TARGET],
            dropna=False,
        )
        .size()
        .reset_index(name="행수")
    )

    # --------------------------------------------------------
    # K. 파싱 이상 행
    # --------------------------------------------------------

    parse_issues = df[[ID, DATE]].copy()

    # 첫 행이 헤더라고 가정한 원본 Excel 행 번호
    parse_issues.insert(
        0,
        "엑셀행",
        np.arange(len(df)) + 2,
    )

    parse_issues["결산일파싱실패"] = dates.isna()
    parse_issues["숫자변환실패수"] = conversion_failures.sum(axis=1)
    parse_issues["무한대수"] = infinite.sum(axis=1)

    issue_mask = (
        parse_issues["결산일파싱실패"]
        | parse_issues["숫자변환실패수"].gt(0)
        | parse_issues["무한대수"].gt(0)
    )

    parse_issues = parse_issues.loc[issue_mask]

    # --------------------------------------------------------
    # L. 변수 사전
    # --------------------------------------------------------

    variable_dictionary = profile[["변수", "분류"]].copy()

    cautions = {
        TARGET: (
            "1/2의 의미와 상태 판정시점 확인 필요. "
            "현재 상태 분류와 미래 부도 예측을 구분할 것."
        ),
        DATE: (
            "예측시점 및 재무정보 이용 가능 시점 확인 필요. "
            "상태별 결산연도 분포 차이를 점검할 것."
        ),
        ID: "기업 식별자. 일반적으로 예측 독립변수에서 제외.",
        "업력": (
            "PDF K1 산식의 기업도산일 참조에 따른 "
            "예측시점 이후 정보 포함 가능성 확인 필요."
        ),
        "순운전자본회전기간": (
            "명칭과 PDF 산식 간 불일치 가능성 확인 필요."
        ),
        AUDIT: (
            "숫자형 연속변수가 아닌 범주형 변수. "
            "미확인 코드의 의미를 임의로 지정하지 말 것."
        ),
    }

    variable_dictionary["주의사항"] = (
        variable_dictionary["변수"]
        .map(cautions)
        .fillna("")
    )

    # --------------------------------------------------------
    # M. 결과 사전
    # --------------------------------------------------------

    tables = {
        "profile": profile,
        "numeric_stats": stats.reset_index(),
        "target_counts": target_counts,
        "industry_target": industry_target.reset_index(),
        "year_target": year_target.reset_index(),
        "year_share_within_target": (
            year_share_within_target.reset_index()
        ),
        "target_share_within_year": (
            target_share_within_year.reset_index()
        ),
        "audit_target": audit_target.reset_index(),
        "identical_columns": identical_table,
        "stats_by_target": stats_by_target,
        "missing_by_target": missing_by_target,
        "company_counts": company_counts,
        "year_industry_target": year_industry_target,
        "parse_issues": parse_issues,
        "variable_dictionary": variable_dictionary,
        "duplicate_rows": df.loc[df.duplicated(keep=False)],
    }

    tables.update(correlation_tables)

    summary = {
        "행수": len(df),
        "열수": len(df.columns),
        "고유기업수": int(df[ID].nunique()),
        "수치분석변수수": len(numeric_features),
        "전체결측셀수": int(df.isna().sum().sum()),
        "완전중복행수": int(df.duplicated().sum()),
        "회사_결산일중복수": int(
            df.duplicated([ID, DATE]).sum()
        ),
        "결산일최소": str(dates.min()),
        "결산일최대": str(dates.max()),
        "결산일파싱실패수": int(dates.isna().sum()),
        "숫자변환실패셀수": int(
            conversion_failures.sum().sum()
        ),
        "무한대셀수": int(infinite.sum().sum()),
        "전체결측변수": x.columns[
            x.notna().sum().eq(0)
        ].tolist(),
        "상수변수_결측제외": x.columns[
            x.nunique(dropna=True).eq(1)
        ].tolist(),
        "완전동일변수쌍": identical_pairs,
        "산업사전미등록코드": sorted(
            set(df[INDUSTRY].dropna()) - set(INDUSTRIES)
        ),
    }

    return x, dates, tables, summary


# ============================================================
# 6. 보기 좋게 출력하기
# ============================================================

def show_table(title, table, n=20):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)

    shown = table if n is None else table.head(n)

    try:
        from IPython.display import display
        display(shown)
    except ImportError:
        print(shown.to_string(index=False))


# ============================================================
# 7. 선택 사항: 탐색 결과 저장
# ============================================================

def save_results(tables, summary, output_dir):
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)

    # 상관행렬은 변수명이 인덱스에 있으므로 함께 저장
    matrix_names = {"pearson_matrix", "spearman_matrix"}

    with pd.ExcelWriter(
        out / "eda_tables.xlsx",
        engine="openpyxl",
    ) as writer:

        for name, table in tables.items():
            include_index = name in matrix_names

            table.to_excel(
                writer,
                sheet_name=name[:31],
                index=include_index,
            )

            table.to_csv(
                out / f"{name}.csv",
                index=include_index,
                encoding="utf-8-sig",
            )

            ws = writer.sheets[name[:31]]
            ws.freeze_panes = "B2" if include_index else "A2"
            ws.auto_filter.ref = ws.dimensions

    (out / "summary.json").write_text(
        json.dumps(
            summary,
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )

    print(f"\n결과 저장 위치: {out.resolve()}")


## 3. 전체 탐색 실행


In [ ]:
# ============================================================
# 데이터 로딩
# ============================================================

print("데이터 크기:", df.shape)

show_table(
    "원본 데이터 앞부분",
    df,
    n=5,
)


# ============================================================
# EDA 실행
# ============================================================

x, dates, tables, summary = explore_data(
    df,
    corr_threshold=CORR_THRESHOLD,
)

print("\n[전체 요약]")
print(
    json.dumps(
        summary,
        ensure_ascii=False,
        indent=2,
    )
)


# ============================================================
# 1. 변수 구성
# ============================================================

show_table(
    "변수별 자료형 / 분류 / 결측 / 고유값",
    tables["profile"],
    n=None,
)

show_table(
    "변수 분류별 개수",
    tables["profile"]
    .groupby("분류", dropna=False)
    .size()
    .reset_index(name="변수수"),
)


# ============================================================
# 2. 결측치
# ============================================================

show_table(
    "결측률 상위 변수",
    tables["profile"].sort_values(
        "결측비율",
        ascending=False,
    ),
    n=30,
)

# 행별 결측률도 확인
row_missing_rate = x.isna().mean(axis=1)

show_table(
    "기업별 수치변수 결측률 요약",
    row_missing_rate.describe(
        percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
    )
    .rename_axis("통계량")
    .reset_index(name="결측비율"),
)

show_table(
    "결측률이 높은 기업",
    df[[ID, INDUSTRY, TARGET, DATE]]
    .assign(수치변수결측비율=row_missing_rate)
    .sort_values(
        "수치변수결측비율",
        ascending=False,
    ),
    n=20,
)


# ============================================================
# 3. 수치 분포 / 0 / 음수 / 극단값
# ============================================================

show_table(
    "수치변수 기술통계",
    tables["numeric_stats"],
    n=20,
)

show_table(
    "0 비율이 높은 변수",
    tables["numeric_stats"].sort_values(
        "0비율_유효값기준",
        ascending=False,
    ),
    n=20,
)

show_table(
    "음수 비율이 높은 변수",
    tables["numeric_stats"].sort_values(
        "음수비율_유효값기준",
        ascending=False,
    ),
    n=20,
)

show_table(
    "IQR 기준 극단값 비율이 높은 변수 — 자동 삭제 대상은 아님",
    tables["numeric_stats"].sort_values(
        "IQR극단값비율",
        ascending=False,
    ),
    n=25,
)


# ============================================================
# 4. 타깃 분포
# ============================================================

show_table(
    "정상/폐업 코드별 행 수",
    tables["target_counts"],
)

show_table(
    "산업별 상태 코드 분포",
    tables["industry_target"],
    n=None,
)

show_table(
    "감사의견별 상태 코드 분포",
    tables["audit_target"],
    n=None,
)


# ============================================================
# 5. 시점 교란 점검
# ============================================================

show_table(
    "결산연도별 상태 코드 수",
    tables["year_target"],
    n=None,
)

show_table(
    "각 상태 코드 내부에서 연도별 구성비",
    tables["year_share_within_target"],
    n=None,
)

show_table(
    "각 연도 내부에서 상태 코드별 구성비",
    tables["target_share_within_year"],
    n=None,
)


# ============================================================
# 6. 상태 코드별 결측 차이
# ============================================================

missing_comparison = (
    tables["missing_by_target"]
    .pivot(
        index="변수",
        columns=TARGET,
        values="결측비율",
    )
)

# 여러 상태 코드 사이의 결측률 최대 차이
missing_comparison["결측률최대차이"] = (
    missing_comparison.max(axis=1)
    - missing_comparison.min(axis=1)
)

show_table(
    "상태 코드에 따라 결측률 차이가 큰 변수",
    missing_comparison
    .sort_values(
        "결측률최대차이",
        ascending=False,
    )
    .reset_index(),
    n=30,
)


# ============================================================
# 7. 주요 재무변수의 코드별 통계
# ============================================================

KEY_VARIABLES = [
    "영업이익률",
    "총자산순이익률(ROA)",
    "ROE",
    "부채비율",
    "자기자본비율",
    "유동비율",
    "차입금의존도",
    "이자보상배수(배)",
    "현금흐름/매출액",
    "종업원수",
    "업력",
]

show_table(
    "주요 변수의 상태 코드별 통계",
    tables["stats_by_target"]
    .loc[
        lambda t: t["변수"].isin(KEY_VARIABLES)
    ]
    .sort_values(["변수", TARGET]),
    n=None,
)

median_comparison = (
    tables["stats_by_target"]
    .pivot(
        index="변수",
        columns=TARGET,
        values="median",
    )
)

show_table(
    "주요 변수의 코드별 중앙값",
    median_comparison
    .reindex(
        [
            col for col in KEY_VARIABLES
            if col in median_comparison.index
        ]
    )
    .reset_index(),
    n=None,
)


# ============================================================
# 8. 중복 변수 / 다중공선성 후보
# ============================================================

show_table(
    "완전히 동일한 변수 쌍",
    tables["identical_columns"],
    n=None,
)

show_table(
    "Pearson 고상관 변수 쌍",
    tables["pearson_highcorr"],
    n=30,
)

show_table(
    "Spearman 고상관 변수 쌍",
    tables["spearman_highcorr"],
    n=30,
)


# ============================================================
# 9. 데이터 구조 / 파싱 오류
# ============================================================

show_table(
    "기업별 관측 행 수",
    tables["company_counts"].sort_values(
        "행수",
        ascending=False,
    ),
    n=10,
)

show_table(
    "파싱 이상 행",
    tables["parse_issues"],
    n=20,
)

show_table(
    "해석 시 주의사항이 있는 변수",
    tables["variable_dictionary"].loc[
        lambda t: t["주의사항"].ne("")
    ],
    n=None,
)


# ============================================================
# 10. 선택 저장
# ============================================================

if SAVE_RESULTS:
    save_results(
        tables,
        summary,
        OUTPUT_DIR,
    )


## 4. 전처리

### 기존 전처리

In [ ]:
# 처음 한 번만 실행
%pip install -U scikit-learn


In [ ]:
  import numpy as np
  import pandas as pd

  from sklearn.base import BaseEstimator, TransformerMixin
  from sklearn.model_selection import train_test_split
  from sklearn.pipeline import Pipeline
  from sklearn.compose import ColumnTransformer
  from sklearn.preprocessing import OneHotEncoder, RobustScaler
  from sklearn.impute import SimpleImputer
  from sklearn.utils.validation import check_is_fitted


  # ============================================================
  # 1. 설정
  # ============================================================

  SEED = 42

  ID_COL = "회사코드"
  TARGET_COL = "정상/폐업"
  DATE_COL = "재무비율기준/결산기준일"

  CATEGORICAL_COLS = [
      "산업코드",
      "감사의견코드",
  ]

  # 예측시점 이후 정보가 포함될 가능성이 있어 우선 제외
  SUSPECTED_LEAKAGE_COLS = ["업력"]

  # 훈련셋에서 결측률이 60%를 초과하는 수치변수 제거
  MISSING_THRESHOLD = 0.60

  # 훈련셋 기준 양쪽 1% 극단값 제한
  LOWER_QUANTILE = 0.01
  UPPER_QUANTILE = 0.99


  # ============================================================
  # 2. 훈련셋에서 수치변수 선택
  #
  # - 결측률 과다 변수
  # - 상수/전체 결측 변수
  # - 완전히 같은 수치변수
  #
  # 해당 판단은 fit()에 전달된 데이터에서만 수행한다.
  # ============================================================

  class NumericColumnSelector(BaseEstimator, TransformerMixin):

      def __init__(self, missing_threshold=0.60):
          self.missing_threshold = missing_threshold

      def fit(self, X, y=None):
          if not isinstance(X, pd.DataFrame):
              raise TypeError("입력은 pandas DataFrame이어야 합니다.")

          self.feature_names_in_ = np.asarray(X.columns, dtype=object)

          self.missing_rates_ = X.isna().mean()

          self.high_missing_cols_ = self.missing_rates_[
              self.missing_rates_ > self.missing_threshold
          ].index.tolist()

          self.constant_cols_ = X.columns[
              X.nunique(dropna=True) <= 1
          ].tolist()

          excluded = set(
              self.high_missing_cols_ + self.constant_cols_
          )

          candidates = [
              col for col in X.columns
              if col not in excluded
          ]

          self.duplicate_mapping_ = {}
          retained = []

          for col in candidates:
              duplicate_of = next(
                  (
                      previous
                      for previous in retained
                      if X[col].equals(X[previous])
                  ),
                  None,
              )

              if duplicate_of is None:
                  retained.append(col)
              else:
                  self.duplicate_mapping_[col] = duplicate_of

          if not retained:
              raise ValueError("선택 후 남는 수치변수가 없습니다.")

          self.selected_columns_ = retained
          return self

      def transform(self, X):
          check_is_fitted(self, "selected_columns_")

          missing = set(self.selected_columns_) - set(X.columns)
          if missing:
              raise ValueError(f"필수 수치변수 누락: {missing}")

          return X.loc[:, self.selected_columns_].copy()

      def get_feature_names_out(self, input_features=None):
          check_is_fitted(self, "selected_columns_")
          return np.asarray(self.selected_columns_, dtype=object)


  # ============================================================
  # 3. 훈련셋 분위수 기반 극단값 제한
  #
  # 결측 여부가 보존되도록 대치 전에 실행한다.
  # 행은 삭제하지 않는다.
  # ============================================================

  class QuantileClipper(BaseEstimator, TransformerMixin):

      def __init__(self, lower=0.01, upper=0.99):
          self.lower = lower
          self.upper = upper

      def fit(self, X, y=None):
          if not 0 <= self.lower < self.upper <= 1:
              raise ValueError("0 <= lower < upper <= 1 이어야 합니다.")

          self.feature_names_in_ = np.asarray(X.columns, dtype=object)

          self.lower_bounds_ = X.quantile(self.lower)
          self.upper_bounds_ = X.quantile(self.upper)

          # 0이 많은 변수에서는 두 분위수가 같아질 수 있다.
          # 그때 전체 값을 같은 값으로 압축하지 않도록 제한을 생략한다.
          usable = self.upper_bounds_ > self.lower_bounds_

          self.lower_bounds_ = self.lower_bounds_.where(
              usable,
              -np.inf,
          )

          self.upper_bounds_ = self.upper_bounds_.where(
              usable,
              np.inf,
          )

          return self

      def transform(self, X):
          check_is_fitted(self, "lower_bounds_")

          return X.clip(
              lower=self.lower_bounds_,
              upper=self.upper_bounds_,
              axis=1,
          )

      def get_feature_names_out(self, input_features=None):
          check_is_fitted(self, "feature_names_in_")
          return self.feature_names_in_


  # ============================================================
  # 4. 수치 결측 여부 표시
  #
  # 훈련셋에서 결측이 없었어도 미래 데이터에는 생길 수 있으므로
  # 선택된 수치변수 전체에 대해 표시 변수를 만든다.
  # ============================================================

  class MissingFlags(BaseEstimator, TransformerMixin):

      def fit(self, X, y=None):
          self.feature_names_in_ = np.asarray(X.columns, dtype=object)
          return self

      def transform(self, X):
          check_is_fitted(self, "feature_names_in_")
          return X.isna().astype(np.float64).to_numpy()

      def get_feature_names_out(self, input_features=None):
          return np.asarray(
              [f"{col}__missing" for col in self.feature_names_in_],
              dtype=object,
          )


  # ============================================================
  # 5. 입력 형태 정리
  #
  # 이 단계는 통계량을 학습하지 않는다.
  # 추후 새로운 기업 데이터를 넣을 때도 같은 함수를 사용한다.
  # ============================================================

  def prepare_features(data):
      required = set(CATEGORICAL_COLS)

      missing = required - set(data.columns)
      if missing:
          raise ValueError(f"범주형 변수 누락: {missing}")

      drop_cols = [
          ID_COL,
          TARGET_COL,
          DATE_COL,
          *SUSPECTED_LEAKAGE_COLS,
      ]

      X = data.drop(
          columns=drop_cols,
          errors="ignore",
      ).copy()

      # 범주형: 의미 미확인 코드는 그대로 유지한다.
      # DU, NS 등의 의미를 임의 추정하거나 순서를 부여하지 않음.
      for col in CATEGORICAL_COLS:
          values = X[col].astype("string").str.strip()
          values = values.mask(values.eq(""), pd.NA)

          # sklearn에서 일관되게 다루도록 np.nan으로 통일
          X[col] = values.astype(object)
          X.loc[values.isna(), col] = np.nan

      numeric_cols = [
          col for col in X.columns
          if col not in CATEGORICAL_COLS
      ]

      for col in numeric_cols:
          X[col] = pd.to_numeric(X[col], errors="coerce")

      X[numeric_cols] = X[numeric_cols].replace(
          [np.inf, -np.inf],
          np.nan,
      )

      return X


  # ============================================================
  # 6. 타깃과 행 정리
  # ============================================================

  data = df.copy()
  data.columns = data.columns.astype(str).str.strip()

  required_columns = {
      ID_COL,
      TARGET_COL,
      DATE_COL,
      *CATEGORICAL_COLS,
  }

  missing_columns = required_columns - set(data.columns)
  if missing_columns:
      raise ValueError(f"필수 컬럼 누락: {missing_columns}")

  # 라벨 결측 행은 지도학습에서 제외
  target_text = (
      data[TARGET_COL]
      .astype("string")
      .str.strip()
      .str.replace(r"\.0$", "", regex=True)
      .replace("", pd.NA)
  )

  print("타깃 결측으로 제외하는 행 수:", target_text.isna().sum())

  data = data.loc[target_text.notna()].copy()
  target_text = target_text.loc[data.index]

  # 완전히 동일한 행만 제거
  duplicate_mask = data.duplicated()
  print("완전 중복으로 제외하는 행 수:", duplicate_mask.sum())

  data = data.loc[~duplicate_mask].copy()
  target_text = target_text.loc[data.index]

  # 기업이 여러 행이면 기업 단위 분할이 필요하므로
  # 같은 기업이 train/test에 섞이지 않도록 여기서는 중단한다.
  company_ids = data[ID_COL].astype("string").str.strip()

  if company_ids.isna().any() or company_ids.eq("").any():
      raise ValueError("회사코드 결측을 먼저 확인해 주세요.")

  if company_ids.duplicated().any():
      raise ValueError(
          "기업당 여러 행이 있습니다. "
          "현재 코드 대신 기업 단위 그룹 분할을 사용해야 합니다."
      )

  observed_labels = set(target_text.unique())

  if observed_labels != {"1", "2"}:
      raise ValueError(
          f"예상하지 못한 타깃 코드: {observed_labels}"
      )

  # 의미를 추정하지 않고 단순히 이진 코드로 변환
  # 0 = 원본 코드 1
  # 1 = 원본 코드 2
  # 양성 클래스 1이 '폐업'이라는 뜻은 아직 아니다.
  TARGET_MAPPING = {"1": 0, "2": 1}

  y = target_text.map(TARGET_MAPPING).astype("int64")
  X = prepare_features(data)

  # 행 인덱스 정렬
  X = X.reset_index(drop=True)
  y = y.reset_index(drop=True)

  metadata = data[
      [ID_COL, DATE_COL]
  ].reset_index(drop=True)

  metadata["결산연도"] = pd.to_datetime(
      metadata[DATE_COL]
      .astype("string")
      .str.replace(r"\.0$", "", regex=True),
      format="%Y%m%d",
      errors="coerce",
  ).dt.year

  numeric_cols = [
      col for col in X.columns
      if col not in CATEGORICAL_COLS
  ]

  print("\n타깃 매핑:", TARGET_MAPPING)
  print("전처리 전 X:", X.shape)
  print("수치형 변수 수:", len(numeric_cols))
  print("범주형 변수:", CATEGORICAL_COLS)


  # ============================================================
  # 7. 훈련 / 검증 / 테스트 분리
  #
  # 60% / 20% / 20%
  # 이 분할은 동일 표본 내부의 비교용이다.
  # 미래 시점 일반화 검증을 보장하지는 않는다.
  # ============================================================

  all_indices = np.arange(len(X))

  train_idx, remaining_idx = train_test_split(
      all_indices,
      test_size=0.40,
      stratify=y,
      random_state=SEED,
  )

  valid_idx, test_idx = train_test_split(
      remaining_idx,
      test_size=0.50,
      stratify=y.iloc[remaining_idx],
      random_state=SEED,
  )

  X_train = X.iloc[train_idx].copy()
  X_valid = X.iloc[valid_idx].copy()
  X_test = X.iloc[test_idx].copy()

  y_train = y.iloc[train_idx].copy()
  y_valid = y.iloc[valid_idx].copy()
  y_test = y.iloc[test_idx].copy()

  meta_train = metadata.iloc[train_idx].copy()
  meta_valid = metadata.iloc[valid_idx].copy()
  meta_test = metadata.iloc[test_idx].copy()


  # ============================================================
  # 8. 전처리 파이프라인
  # ============================================================

  # 수치 값:
  # 극단값 제한 → 중앙값 대치 → RobustScaler
  numeric_values_pipeline = Pipeline([
      (
          "clip",
          QuantileClipper(
              lower=LOWER_QUANTILE,
              upper=UPPER_QUANTILE,
          ),
      ),
      (
          "imputer",
          SimpleImputer(strategy="median"),
      ),
      (
          "scaler",
          RobustScaler(),
      ),
  ])

  # 선택한 수치변수를 값과 결측 여부 두 갈래로 처리
  numeric_branches = ColumnTransformer(
      transformers=[
          (
              "values",
              numeric_values_pipeline,
              slice(None),
          ),
          (
              "missing",
              MissingFlags(),
              slice(None),
          ),
      ],
      remainder="drop",
      verbose_feature_names_out=True,
  )

  numeric_pipeline = Pipeline([
      (
          "select",
          NumericColumnSelector(
              missing_threshold=MISSING_THRESHOLD,
          ),
      ),
      (
          "branches",
          numeric_branches,
      ),
  ])

  # 산업코드와 감사의견에는 임의의 대소 관계를 부여하지 않는다.
  # 결측은 별도 범주로 처리.
  # 훈련 중 보지 못한 새 범주는 오류 대신 모두 0으로 처리된다.
  categorical_pipeline = Pipeline([
      (
          "imputer",
          SimpleImputer(
              strategy="constant",
              fill_value="__MISSING__",
              keep_empty_features=True,
          ),
      ),
      (
          "onehot",
          OneHotEncoder(
              handle_unknown="ignore",
              sparse_output=False,
              dtype=np.float64,
          ),
      ),
  ])

  preprocessor = ColumnTransformer(
      transformers=[
          (
              "numeric",
              numeric_pipeline,
              numeric_cols,
          ),
          (
              "categorical",
              categorical_pipeline,
              CATEGORICAL_COLS,
          ),
      ],
      remainder="drop",
      verbose_feature_names_out=True,
  )


  # ============================================================
  # 9. 훈련셋에만 fit
  # ============================================================

  X_train_array = preprocessor.fit_transform(X_train, y_train)

  # 검증/테스트에서는 fit하지 않음!
  X_valid_array = preprocessor.transform(X_valid)
  X_test_array = preprocessor.transform(X_test)

  feature_names = preprocessor.get_feature_names_out()

  X_train_processed = pd.DataFrame(
      X_train_array,
      index=X_train.index,
      columns=feature_names,
  )

  X_valid_processed = pd.DataFrame(
      X_valid_array,
      index=X_valid.index,
      columns=feature_names,
  )

  X_test_processed = pd.DataFrame(
      X_test_array,
      index=X_test.index,
      columns=feature_names,
  )


  # ============================================================
  # 10. 결과 검증
  # ============================================================

  for name, processed in [
      ("train", X_train_processed),
      ("valid", X_valid_processed),
      ("test", X_test_processed),
  ]:
      assert np.isfinite(processed.to_numpy()).all(), (
          f"{name}에 결측 또는 무한대가 남았습니다."
      )

      assert processed.columns.equals(
          X_train_processed.columns
      ), f"{name}의 열 구성이 다릅니다."

  assert set(meta_train[ID_COL]).isdisjoint(
      set(meta_valid[ID_COL])
  )

  assert set(meta_train[ID_COL]).isdisjoint(
      set(meta_test[ID_COL])
  )

  assert set(meta_valid[ID_COL]).isdisjoint(
      set(meta_test[ID_COL])
  )

  selector = (
      preprocessor
      .named_transformers_["numeric"]
      .named_steps["select"]
  )

  print("\n[전처리 완료]")
  print("훈련:", X_train_processed.shape)
  print("검증:", X_valid_processed.shape)
  print("테스트:", X_test_processed.shape)

  print("\n결측률 과다로 제거:")
  print(selector.high_missing_cols_)

  print("\n상수 또는 전체 결측으로 제거:")
  print(selector.constant_cols_)

  print("\n중복 제거 변수 → 남긴 변수:")
  print(selector.duplicate_mapping_)

  print("\n남긴 수치변수 수:")
  print(len(selector.selected_columns_))

  print("\n최종 독립변수 수:")
  print(len(feature_names))

  print("\n훈련 타깃 비율:")
  print(y_train.value_counts(normalize=True).sort_index())

  print("\n전처리 결과 앞부분:")
  print(X_train_processed.iloc[:5, :12])


### 독립변수 16개 추출

In [ ]:
# ============================================================
# 148개 전처리 결과에서 16개 변수만 뽑아 df16 만들기
# (위 전처리 셀을 먼저 실행한 뒤 이 셀 실행)
# ============================================================
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler

FEATURES_16 = [
    '영업이익률', '재료비/매출액', '감가상각률', '금융비용대총부채비율',
    '금융비용대총비용비율', 'EBITDA/총차입금', '차입금의존도',
    '장단기대여금/총자산', '법인세부담률', '순운전자본대총자본비율',
    '유동자산구성비율', '유보액대총자산비율', '재고자산대유동자산비율',
    '매입채무회전기간', '자본금회전율(회)', '업력',
]
feat15 = [c for c in FEATURES_16 if c != '업력']   # 업력은 148개 전처리에서 빠져 있음

# 1) 15개: 처리된 열 이름에서 값 열(values__)만 정확히 찾기
def find_col(processed, name):
    hits = [c for c in processed.columns if c.endswith("values__" + name)]
    assert len(hits) == 1, (name, hits)
    return hits[0]

col_map = {name: find_col(X_train_processed, name) for name in feat15}

# 2) 업력(K1): 148개 버전에서 제외됐으므로 같은 방식으로 따로 처리 (훈련셋에만 fit)
k1 = (
    pd.to_numeric(data["업력"], errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
    .reset_index(drop=True)
    .to_frame("업력")
)

k1_pipe = Pipeline([
    ("clip", QuantileClipper(lower=LOWER_QUANTILE, upper=UPPER_QUANTILE)),
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler()),
])
k1_pipe.fit(k1.iloc[train_idx])

def build16(processed, k1_part):
    out = processed[[col_map[c] for c in feat15]].copy()
    out.columns = feat15
    out["업력"] = k1_pipe.transform(k1_part).ravel()
    return out[FEATURES_16]

X16_train = build16(X_train_processed, k1.iloc[train_idx])
X16_valid = build16(X_valid_processed, k1.iloc[valid_idx])
X16_test  = build16(X_test_processed,  k1.iloc[test_idx])

# 3) df16: 전체를 하나로 (원래 행 순서, 타깃 + 식별/연도 포함)
df16 = pd.concat([
    X16_train.assign(**{TARGET_COL: y_train}),
    X16_valid.assign(**{TARGET_COL: y_valid}),
    X16_test.assign(**{TARGET_COL: y_test}),
]).sort_index()

df16.insert(0, ID_COL, metadata.loc[df16.index, ID_COL].values)
df16["결산연도"] = metadata.loc[df16.index, "결산연도"].values

print(df16.shape)        # (3140, 19)
print("결측/무한대:", int(df16[FEATURES_16].isna().sum().sum()))
df16.head()

## 5. 모델링

### xgboost

In [ ]:
%pip install -U xgboost scikit-learn matplotlib


#### 데이터 분할

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    log_loss,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    PrecisionRecallDisplay,
)

from xgboost import XGBClassifier

# ============================================================
# df16 분할 (148개 버전과 동일한 분할 방식)
# ============================================================
SEED = 42

# 148개 버전의 분할을 보관 (동일성 검증용)
ref_train_idx, ref_valid_idx, ref_test_idx = (
    train_idx.copy(), valid_idx.copy(), test_idx.copy()
)

assert df16.index.equals(X.index), "df16 행 순서가 원본과 다릅니다."

X16 = df16[FEATURES_16].copy()
y16 = df16[TARGET_COL].astype(int).copy()
meta16 = df16[[ID_COL, "결산연도"]].copy()
industry16 = X.loc[df16.index, "산업코드"]     # 분포 확인용 (모델 입력 아님)

assert len(X16) == len(y16) == len(meta16)
assert set(y16.unique()) == {0, 1}

indices = np.arange(len(X16))

train_idx, temp_idx = train_test_split(
    indices, test_size=0.40, random_state=SEED,
    shuffle=True, stratify=y16,
)

valid_idx, test_idx = train_test_split(
    temp_idx, test_size=0.50, random_state=SEED,
    shuffle=True, stratify=y16.iloc[temp_idx],
)

# 148개 버전과 행이 완전히 같은지 검증
assert np.array_equal(train_idx, ref_train_idx)
assert np.array_equal(valid_idx, ref_valid_idx)
assert np.array_equal(test_idx, ref_test_idx)
print("148개 버전과 동일한 분할 확인")

X_train, X_valid, X_test = (X16.iloc[i].copy() for i in (train_idx, valid_idx, test_idx))
y_train, y_valid, y_test = (y16.iloc[i].copy() for i in (train_idx, valid_idx, test_idx))
meta_train, meta_valid, meta_test = (meta16.iloc[i].copy() for i in (train_idx, valid_idx, test_idx))

assert set(train_idx).isdisjoint(valid_idx)
assert set(train_idx).isdisjoint(test_idx)
assert set(valid_idx).isdisjoint(test_idx)
assert len(train_idx) + len(valid_idx) + len(test_idx) == len(X16)

for left, right in [(meta_train, meta_valid), (meta_train, meta_test), (meta_valid, meta_test)]:
    assert set(left[ID_COL]).isdisjoint(set(right[ID_COL])), "같은 기업이 여러 세트에 포함됐습니다."

split_summary = pd.DataFrame({
    "데이터": ["train", "valid", "test"],
    "행수": [len(y_train), len(y_valid), len(y_test)],
    "원본코드1_개수": [y_train.eq(0).sum(), y_valid.eq(0).sum(), y_test.eq(0).sum()],
    "원본코드2_개수": [y_train.eq(1).sum(), y_valid.eq(1).sum(), y_test.eq(1).sum()],
    "원본코드2_비율": [y_train.mean(), y_valid.mean(), y_test.mean()],
})
print("\n[분할 결과]")
print(split_summary.to_string(index=False))

industry_distribution = pd.DataFrame({
    "train": industry16.iloc[train_idx].value_counts(normalize=True),
    "valid": industry16.iloc[valid_idx].value_counts(normalize=True),
    "test":  industry16.iloc[test_idx].value_counts(normalize=True),
}).fillna(0).sort_index()
print("\n[산업별 구성비]")
print(industry_distribution.round(3))

year_distribution = pd.DataFrame({
    "train": meta_train["결산연도"].value_counts(normalize=True, dropna=False),
    "valid": meta_valid["결산연도"].value_counts(normalize=True, dropna=False),
    "test":  meta_test["결산연도"].value_counts(normalize=True, dropna=False),
}).fillna(0).sort_index()
print("\n[결산연도별 구성비]")
print(year_distribution.round(3))

#### 훈련

In [ ]:
# df16 기준 입력 (이미 극단값 제한 / 대치 / 스케일링 완료된 16개 변수)
X_train_xgb = X_train[FEATURES_16]
X_valid_xgb = X_valid[FEATURES_16]
X_test_xgb  = X_test[FEATURES_16]

xgb_model = XGBClassifier(
    objective="binary:logistic",

    # 조기 종료가 있으므로 상한을 넉넉하게 설정
    n_estimators=2000,
    learning_rate=0.03,

    # 트리 복잡도 제한
    max_depth=3,
    min_child_weight=5,

    # 행 / 변수 샘플링
    subsample=0.8,
    colsample_bytree=0.8,

    # 규제
    reg_alpha=0.1,
    reg_lambda=5.0,

    # 기본 모델에서는 가중치 미조정
    scale_pos_weight=1.0,

    tree_method="hist",
    eval_metric="logloss",
    early_stopping_rounds=100,

    random_state=SEED,
    n_jobs=-1,
    importance_type="gain",
)

xgb_model.fit(
    X_train_xgb,
    y_train,
    eval_set=[
        (X_train_xgb, y_train),
        (X_valid_xgb, y_valid),
    ],
    verbose=100,
)

# best_iteration은 0부터 시작
print("\n최적 트리 개수:", xgb_model.best_iteration + 1)
print("최적 검증 Log loss:", xgb_model.best_score)

#### 검증

In [ ]:
THRESHOLD = 0.5

probabilities = {
    "train": xgb_model.predict_proba(X_train_xgb)[:, 1],
    "valid": xgb_model.predict_proba(X_valid_xgb)[:, 1],
    "test": xgb_model.predict_proba(X_test_xgb)[:, 1],
}

targets = {
    "train": y_train,
    "valid": y_valid,
    "test": y_test,
}


def calculate_metrics(y_true, y_probability, threshold=0.5):
    y_pred = (y_probability >= threshold).astype(int)

    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Balanced accuracy": balanced_accuracy_score(y_true, y_pred),
        "Precision_code2": precision_score(y_true, y_pred, zero_division=0),
        "Recall_code2": recall_score(y_true, y_pred, zero_division=0),
        "F1_code2": f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, y_probability),
        "Average precision": average_precision_score(y_true, y_probability),
        "Log loss": log_loss(y_true, y_probability, labels=[0, 1]),
    }


results = pd.DataFrame({
    name: calculate_metrics(
        targets[name],
        probabilities[name],
        threshold=THRESHOLD,
    )
    for name in ["train", "valid", "test"]
}).T

print("\n[성능 비교]")
print(results.round(4).to_string())

test_pred = (probabilities["test"] >= THRESHOLD).astype(int)

print("\n[테스트 분류 보고서]")
print(
    classification_report(
        y_test,
        test_pred,
        labels=[0, 1],
        target_names=["원본 코드 1", "원본 코드 2"],
        digits=4,
        zero_division=0,
    )
)

print("\n[테스트 혼동행렬]")
print(
    pd.DataFrame(
        confusion_matrix(y_test, test_pred, labels=[0, 1]),
        index=["실제 원본코드1", "실제 원본코드2"],
        columns=["예측 원본코드1", "예측 원본코드2"],
    )
)

### vqc(8qubit)

#### 회로 세팅

In [ ]:
import numpy as np
import pandas as pd

from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA


# ============================================================
# 1. 설정
# ============================================================

N_QUBITS = 8
SEED = 42

# 값이 클수록 tanh 포화가 완만해짐
# 테스트셋을 보고 조정하지 말고, 필요하면 검증셋에서 선택
ANGLE_SCALE = 2.0


# ============================================================
# 2. 입력 확인
# ============================================================

train_array = np.asarray(X_train_xgb, dtype=np.float64)
valid_array = np.asarray(X_valid_xgb, dtype=np.float64)
test_array = np.asarray(X_test_xgb, dtype=np.float64)

for name, array in [
    ("train", train_array),
    ("valid", valid_array),
    ("test", test_array),
]:
    if array.ndim != 2:
        raise ValueError(f"{name}: 2차원 배열이어야 합니다.")

    if not np.isfinite(array).all():
        raise ValueError(f"{name}: 결측 또는 무한대가 있습니다.")

if not (
    train_array.shape[1]
    == valid_array.shape[1]
    == test_array.shape[1]
):
    raise ValueError("훈련·검증·테스트의 변수 수가 다릅니다.")


# ============================================================
# 3. 훈련셋에서 상수 변수 제거
#
# 예: 항상 0인 결측 표시 변수
# ============================================================

q_constant_filter = VarianceThreshold(threshold=0.0)

train_selected = q_constant_filter.fit_transform(train_array)
valid_selected = q_constant_filter.transform(valid_array)
test_selected = q_constant_filter.transform(test_array)

max_components = min(
    train_selected.shape[0] - 1,
    train_selected.shape[1],
)

if not 1 <= N_QUBITS <= max_components:
    raise ValueError(
        f"N_QUBITS는 1~{max_components} 사이여야 합니다."
    )


# ============================================================
# 4. PCA 전 표준화
#
# 기존 RobustScaler가 잘못됐다는 뜻이 아님.
# PCA에서 변수별 분산 차이의 영향을 조절하기 위한 추가 단계.
#
# 원핫/결측 표시처럼 희소한 이진변수도 확대될 수 있으므로
# 이후 표준화 유무를 검증셋에서 비교할 수 있음.
# ============================================================

q_input_scaler = StandardScaler()

train_scaled = q_input_scaler.fit_transform(train_selected)
valid_scaled = q_input_scaler.transform(valid_selected)
test_scaled = q_input_scaler.transform(test_selected)


# ============================================================
# 5. PCA로 소수 특징에 압축
#
# y를 사용하지 않는 비지도 차원 축소
# ============================================================

q_pca = PCA(
    n_components=N_QUBITS,
    svd_solver="full",
    random_state=SEED,
)

Z_train = q_pca.fit_transform(train_scaled)
Z_valid = q_pca.transform(valid_scaled)
Z_test = q_pca.transform(test_scaled)

print("성분별 설명 분산 비율:")
print(np.round(q_pca.explained_variance_ratio_, 4))

print(
    "누적 설명 분산 비율:",
    round(q_pca.explained_variance_ratio_.sum(), 4),
)

# 설명 분산 비율이 높다고 분류 정보가 반드시 잘 보존되는 것은 아님.
# 반대로 낮더라도 분류에 필요한 정보 일부를 담을 수 있으므로
# 최종 판단은 검증 분류 성능으로 해야 함.


# ============================================================
# 6. PCA 성분을 훈련셋 기준으로 표준화
#
# 각 성분의 입력각 크기가 지나치게 달라지는 것을 방지
# ============================================================

q_component_scaler = StandardScaler()

Z_train_standard = q_component_scaler.fit_transform(Z_train)
Z_valid_standard = q_component_scaler.transform(Z_valid)
Z_test_standard = q_component_scaler.transform(Z_test)


# ============================================================
# 7. 양자회로 입력각으로 변환
#
# theta = pi * tanh(z / ANGLE_SCALE)
#
# 단순 선형 각도 변환에서 큰 값이 반복적인 회전으로
# 겹치는 문제를 줄이기 위한 유계 변환.
# 단, 극단 영역의 차이가 압축되는 단점은 있음.
# ============================================================

def to_quantum_angles(z, scale=ANGLE_SCALE):
    if scale <= 0:
        raise ValueError("scale은 양수여야 합니다.")

    return np.pi * np.tanh(
        np.asarray(z, dtype=np.float64) / scale
    )


X_train_q = to_quantum_angles(Z_train_standard)
X_valid_q = to_quantum_angles(Z_valid_standard)
X_test_q = to_quantum_angles(Z_test_standard)


# ============================================================
# 8. 데이터프레임으로 정리
# ============================================================

angle_columns = [
    f"theta_{i}"
    for i in range(N_QUBITS)
]

X_train_q_df = pd.DataFrame(
    X_train_q,
    index=X_train.index,
    columns=angle_columns,
)

X_valid_q_df = pd.DataFrame(
    X_valid_q,
    index=X_valid.index,
    columns=angle_columns,
)

X_test_q_df = pd.DataFrame(
    X_test_q,
    index=X_test.index,
    columns=angle_columns,
)


# ============================================================
# 9. 결과 검증
# ============================================================

for name, array in [
    ("train", X_train_q),
    ("valid", X_valid_q),
    ("test", X_test_q),
]:
    assert np.isfinite(array).all()
    assert array.shape[1] == N_QUBITS
    assert np.abs(array).max() <= np.pi

    print(
        f"{name}: shape={array.shape}, "
        f"각도 범위=({array.min():.4f}, {array.max():.4f})"
    )

# 타깃은 기존 것을 그대로 사용
assert len(X_train_q) == len(y_train)
assert len(X_valid_q) == len(y_valid)
assert len(X_test_q) == len(y_test)

print("\n양자회로 입력 예시:")
print(X_train_q_df.head())


#### 훈련

In [ ]:
%pip install pennylane torch scikit-learn

import copy
import random
import time

import numpy as np
import pandas as pd
import pennylane as qml
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    log_loss,
    classification_report,
    confusion_matrix,
)


# ============================================================
# 설정
# ============================================================

SEED = 42
N_QUBITS = X_train_q.shape[1]
N_LAYERS = 2

BATCH_SIZE = 64
MAX_EPOCHS = 100
PATIENCE = 15
LEARNING_RATE = 0.01

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# 소규모 시뮬레이터 실험이므로 CPU 사용
DEVICE = torch.device("cpu")
DTYPE = torch.float64


# ============================================================
# 입력 확인
# ============================================================

for name, features, labels in [
    ("train", X_train_q, y_train),
    ("valid", X_valid_q, y_valid),
    ("test", X_test_q, y_test),
]:
    features = np.asarray(features)
    labels = np.asarray(labels).reshape(-1)

    assert features.ndim == 2
    assert features.shape[1] == N_QUBITS
    assert len(features) == len(labels)
    assert np.isfinite(features).all()
    assert np.isin(labels, [0, 1]).all()


# ============================================================
# 양자회로
#
# 입력: 기업당 N_QUBITS개 회전각
# 출력: 각 큐비트의 Pauli-Z 기대값
# ============================================================

quantum_device = qml.device(
    "default.qubit",
    wires=N_QUBITS,
    shots=None,  # 샷 샘플링 없이 기대값 계산
)


@qml.qnode(
    quantum_device,
    interface="torch",
    diff_method="backprop",
)
def quantum_circuit(inputs, weights):

    # 입력값을 각 큐비트의 RY 회전각으로 인코딩
    qml.AngleEmbedding(
        features=inputs,
        wires=range(N_QUBITS),
        rotation="Y",
    )

    # 학습 가능한 회전 게이트 + 얽힘 게이트
    qml.StronglyEntanglingLayers(
        weights=weights,
        wires=range(N_QUBITS),
    )

    return [
        qml.expval(qml.PauliZ(wire))
        for wire in range(N_QUBITS)
    ]


class VQCClassifier(nn.Module):

    def __init__(self, n_qubits, n_layers):
        super().__init__()

        weight_shapes = {
            "weights": qml.StronglyEntanglingLayers.shape(
                n_layers=n_layers,
                n_wires=n_qubits,
            )
        }

        self.quantum_layer = qml.qnn.TorchLayer(
            quantum_circuit,
            weight_shapes=weight_shapes,
        )

        # 여러 큐비트의 측정값을 하나의 분류 logit으로 결합
        self.output_layer = nn.Linear(n_qubits, 1)

    def forward(self, inputs):
        measurements = self.quantum_layer(inputs)

        # BCEWithLogitsLoss를 쓰므로 sigmoid는 여기서 적용하지 않음
        logits = self.output_layer(measurements)

        return logits.squeeze(-1)


vqc_model = VQCClassifier(
    n_qubits=N_QUBITS,
    n_layers=N_LAYERS,
).to(device=DEVICE, dtype=DTYPE)

print(vqc_model)

print(
    "\n학습 파라미터 수:",
    sum(
        parameter.numel()
        for parameter in vqc_model.parameters()
        if parameter.requires_grad
    ),
)


In [ ]:
def make_feature_tensor(array):
    return torch.as_tensor(
        np.asarray(array, dtype=np.float64),
        dtype=DTYPE,
        device=DEVICE,
    )


def make_target_tensor(labels):
    return torch.as_tensor(
        np.asarray(labels, dtype=np.float64).reshape(-1),
        dtype=DTYPE,
        device=DEVICE,
    )


train_tensor = make_feature_tensor(X_train_q)
valid_tensor = make_feature_tensor(X_valid_q)

train_target = make_target_tensor(y_train)
valid_target = make_target_tensor(y_valid)

loader_generator = torch.Generator()
loader_generator.manual_seed(SEED)

train_loader = DataLoader(
    TensorDataset(train_tensor, train_target),
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=loader_generator,
)

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    vqc_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=1e-4,
)


# ============================================================
# 評価用: バッチ単位でロジットを計算
# ============================================================

def predict_logits(model, features, batch_size=128):
    model.eval()

    feature_tensor = make_feature_tensor(features)
    outputs = []

    with torch.no_grad():
        for batch in feature_tensor.split(batch_size):
            outputs.append(model(batch))

    return torch.cat(outputs, dim=0)


# ============================================================
# 学習
# ============================================================

best_valid_loss = float("inf")
best_state = None
best_epoch = 0
waiting = 0
history = []

start_time = time.perf_counter()

for epoch in range(1, MAX_EPOCHS + 1):
    vqc_model.train()

    for batch_features, batch_labels in train_loader:
        optimizer.zero_grad()

        logits = vqc_model(batch_features)
        loss = criterion(logits, batch_labels)

        loss.backward()
        optimizer.step()

    # 現在の同じモデルで訓練・検証損失を評価
    train_logits = predict_logits(vqc_model, X_train_q)
    valid_logits = predict_logits(vqc_model, X_valid_q)

    with torch.no_grad():
        train_loss = criterion(
            train_logits, train_target
        ).item()

        valid_loss = criterion(
            valid_logits, valid_target
        ).item()

    train_probability = torch.sigmoid(
        train_logits
    ).cpu().numpy()

    valid_probability = torch.sigmoid(
        valid_logits
    ).cpu().numpy()

    train_auc = roc_auc_score(
        y_train, train_probability
    )

    valid_auc = roc_auc_score(
        y_valid, valid_probability
    )

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "valid_loss": valid_loss,
        "train_auc": train_auc,
        "valid_auc": valid_auc,
    })

    if valid_loss < best_valid_loss - 1e-5:
        best_valid_loss = valid_loss
        best_state = copy.deepcopy(vqc_model.state_dict())
        best_epoch = epoch
        waiting = 0
    else:
        waiting += 1

    if epoch == 1 or epoch % 5 == 0:
        print(
            f"Epoch {epoch:03d} | "
            f"train loss={train_loss:.4f} | "
            f"valid loss={valid_loss:.4f} | "
            f"train AUC={train_auc:.4f} | "
            f"valid AUC={valid_auc:.4f}"
        )

    if waiting >= PATIENCE:
        print(f"\n조기 종료: epoch {epoch}")
        break

if best_state is None:
    raise RuntimeError(
        "정상적인 모델이 저장되지 않았습니다. 손실값을 확인하세요."
    )

# 최적 검증 모델 복원
vqc_model.load_state_dict(best_state)
vqc_model.eval()

training_seconds = time.perf_counter() - start_time
vqc_history = pd.DataFrame(history)

print("\n최적 epoch:", best_epoch)
print("최적 검증 loss:", round(best_valid_loss, 4))
print("학습·검증 소요 시간(초):", round(training_seconds, 2))


#### 검증

In [ ]:
def predict_probability(model, features):
    logits = predict_logits(model, features)

    return torch.sigmoid(logits).cpu().numpy()


def evaluate_binary_classifier(
    labels,
    probabilities,
    threshold=0.5,
):
    labels = np.asarray(labels).reshape(-1)
    probabilities = np.asarray(
        probabilities,
        dtype=float,
    ).reshape(-1)

    if len(labels) != len(probabilities):
        raise ValueError("라벨과 예측확률의 길이가 다릅니다.")

    if not np.isfinite(probabilities).all():
        raise ValueError("예측확률에 NaN 또는 무한대가 있습니다.")

    predictions = (
        probabilities >= threshold
    ).astype(int)

    metrics = {
        "Accuracy": accuracy_score(labels, predictions),

        "Balanced accuracy": balanced_accuracy_score(
            labels, predictions
        ),

        "Precision_code2": precision_score(
            labels, predictions, zero_division=0
        ),

        "Recall_code2": recall_score(
            labels, predictions, zero_division=0
        ),

        "F1_code2": f1_score(
            labels, predictions, zero_division=0
        ),

        "ROC-AUC": roc_auc_score(
            labels, probabilities
        ),

        "Average precision": average_precision_score(
            labels, probabilities
        ),

        "Log loss": log_loss(
            labels,
            probabilities,
            labels=[0, 1],
        ),
    }

    return metrics, predictions


THRESHOLD = 0.5

vqc_probabilities = {}
vqc_predictions = {}
metric_rows = {}

for split, features, labels in [
    ("train", X_train_q, y_train),
    ("valid", X_valid_q, y_valid),
    ("test", X_test_q, y_test),
]:
    probability = predict_probability(
        vqc_model, features
    )

    metrics, prediction = evaluate_binary_classifier(
        labels,
        probability,
        threshold=THRESHOLD,
    )

    vqc_probabilities[split] = probability
    vqc_predictions[split] = prediction
    metric_rows[split] = metrics

vqc_results = pd.DataFrame.from_dict(
    metric_rows,
    orient="index",
)

print("\n[VQC 성능 비교]")
print(vqc_results.round(4).to_string())

print("\n[VQC 테스트 분류 보고서]")
print(
    classification_report(
        y_test,
        vqc_predictions["test"],
        labels=[0, 1],
        target_names=[
            "원본 코드 1",
            "원본 코드 2",
        ],
        digits=4,
        zero_division=0,
    )
)

vqc_confusion = pd.DataFrame(
    confusion_matrix(
        y_test,
        vqc_predictions["test"],
        labels=[0, 1],
    ),
    index=[
        "실제 원본코드1",
        "실제 원본코드2",
    ],
    columns=[
        "예측 원본코드1",
        "예측 원본코드2",
    ],
)

print("\n[VQC 테스트 혼동행렬]")
print(vqc_confusion)


### xgboost(pca 변수 8개)

In [ ]:
# ============================================================
# XGBoost + PCA 8개
# ============================================================

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from xgboost import XGBClassifier

# ------------------------------------------------------------
# 1. PCA 적용 전 스케일링
#    훈련 데이터에만 fit하고, 검증/테스트에는 transform만 적용
# ------------------------------------------------------------

pca_scaler = StandardScaler()

X_train_scaled = pca_scaler.fit_transform(X_train_xgb)
X_valid_scaled = pca_scaler.transform(X_valid_xgb)
X_test_scaled = pca_scaler.transform(X_test_xgb)

# ------------------------------------------------------------
# 2. PCA로 8개 주성분 추출
# ------------------------------------------------------------

N_COMPONENTS = 8

pca = PCA(
    n_components=N_COMPONENTS,
    svd_solver="full",
)

X_train_pca = pca.fit_transform(X_train_scaled)
X_valid_pca = pca.transform(X_valid_scaled)
X_test_pca = pca.transform(X_test_scaled)

print("[PCA 설명분산비율]")
print(pca.explained_variance_ratio_)

print(
    "누적 설명분산비율:",
    pca.explained_variance_ratio_.sum()
)

print("\n[PCA 데이터 크기]")
print("Train:", X_train_pca.shape)
print("Valid:", X_valid_pca.shape)
print("Test :", X_test_pca.shape)

# ------------------------------------------------------------
# 3. XGBoost 학습
#    기존 코드에서 사용하던 설정과 동일하게 맞출 것
# ------------------------------------------------------------

xgb_pca = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
)

xgb_pca.fit(
    X_train_pca,
    y_train,
)

# ------------------------------------------------------------
# 4. 검증 / 테스트 예측
# ------------------------------------------------------------

y_valid_pred_pca = xgb_pca.predict(X_valid_pca)
y_valid_prob_pca = xgb_pca.predict_proba(X_valid_pca)[:, 1]

y_test_pred_pca = xgb_pca.predict(X_test_pca)
y_test_prob_pca = xgb_pca.predict_proba(X_test_pca)[:, 1]

# ------------------------------------------------------------
# 5. 성능 평가 함수
#    기존 노트북에 동일한 평가 함수가 있다면 이 부분은 생략 가능
# ------------------------------------------------------------

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    log_loss,
)

def evaluate_xgb_pca(y_true, y_pred, y_prob):
    return pd.Series({
        "Accuracy": accuracy_score(y_true, y_pred),
        "Balanced accuracy": balanced_accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, y_prob),
        "Average precision": average_precision_score(y_true, y_prob),
        "Log loss": log_loss(y_true, y_prob),
    })

results_pca = pd.DataFrame({
    "valid": evaluate_xgb_pca(
        y_valid, y_valid_pred_pca, y_valid_prob_pca
    ),
    "test": evaluate_xgb_pca(
        y_test, y_test_pred_pca, y_test_prob_pca
    ),
}).T

print("\n[XGBoost + PCA 8개 성능]")
display(results_pca)

### vqc(8qubit + drc)

In [ ]:

import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import pennylane as qml

from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    log_loss,
    confusion_matrix,
    classification_report,
)

# ==================================================
# 1. 하이퍼파라미터
# ==================================================

N_QUBITS = X_train_q.shape[1]  # PCA 입력 차원 = 8
N_LAYERS = 2                   # 재업로딩 블록 안의 양자 레이어 수
N_REUPLOADS = 2                # 데이터 인코딩 반복 횟수

BATCH_SIZE = 64
MAX_EPOCHS = 100
PATIENCE = 15
LEARNING_RATE = 0.01

DEVICE = torch.device("cpu")
DTYPE = torch.float64

assert N_QUBITS == 8, (
    f"이 실험은 8큐비트 기준이야. 현재 입력 차원: {N_QUBITS}"
)

torch.manual_seed(42)
np.random.seed(42)

# ==================================================
# 2. 양자 회로
# ==================================================

qdev = qml.device(
    "default.qubit",
    wires=N_QUBITS,
    shots=None
)

@qml.qnode(
    qdev,
    interface="torch",
    diff_method="backprop"
)
def reuploading_circuit(inputs, weights):

    wires = range(N_QUBITS)

    # 데이터를 2회 인코딩
    for r in range(N_REUPLOADS):

        # PCA 성분을 Y축 회전으로 인코딩
        qml.AngleEmbedding(
            features=inputs,
            wires=wires,
            rotation="Y"
        )

        # 각 인코딩 뒤에 학습 가능한 양자 회로 적용
        qml.StronglyEntanglingLayers(
            weights=weights[r],
            wires=wires
        )

    # 각 큐비트의 Pauli-Z 기대값
    return [
        qml.expval(qml.PauliZ(i))
        for i in wires
    ]


# ==================================================
# 3. TorchLayer 및 VQC 분류기
# ==================================================

weight_shapes = {
    "weights": (
        N_REUPLOADS,
        N_LAYERS,
        N_QUBITS,
        3
    )
}

quantum_layer = qml.qnn.TorchLayer(
    reuploading_circuit,
    weight_shapes
)


class VQCReuploadClassifier(nn.Module):

    def __init__(self, n_qubits=8):
        super().__init__()

        self.quantum_layer = qml.qnn.TorchLayer(
            reuploading_circuit,
            weight_shapes
        )

        # 양자 측정값 8개를 받아 이진분류 logit 출력
        self.classifier = nn.Linear(
            n_qubits,
            1
        )

    def forward(self, x):
        quantum_output = self.quantum_layer(x)
        logits = self.classifier(quantum_output)

        return logits.squeeze(-1)


# ==================================================
# 4. 데이터 변환
# ==================================================

def to_tensor(x):
    return torch.as_tensor(
        np.asarray(x),
        dtype=DTYPE,
        device=DEVICE
    )


Xtr = to_tensor(X_train_q)
Xva = to_tensor(X_valid_q)
Xte = to_tensor(X_test_q)

ytr = to_tensor(y_train).reshape(-1)
yva = to_tensor(y_valid).reshape(-1)
yte = to_tensor(y_test).reshape(-1)

train_loader = DataLoader(
    TensorDataset(Xtr, ytr),
    batch_size=BATCH_SIZE,
    shuffle=True
)

valid_loader = DataLoader(
    TensorDataset(Xva, yva),
    batch_size=BATCH_SIZE,
    shuffle=False
)

# ==================================================
# 5. 모델 학습
# ==================================================

vqc_reupload = VQCReuploadClassifier(
    n_qubits=N_QUBITS
).to(device=DEVICE, dtype=DTYPE)

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    vqc_reupload.parameters(),
    lr=LEARNING_RATE
)

best_val_loss = float("inf")
best_state = None
patience_counter = 0

for epoch in range(MAX_EPOCHS):

    # ----- Training -----
    vqc_reupload.train()
    train_loss_sum = 0.0
    train_count = 0

    for xb, yb in train_loader:
        optimizer.zero_grad()

        logits = vqc_reupload(xb)
        loss = criterion(logits, yb)

        loss.backward()
        optimizer.step()

        train_loss_sum += loss.item() * len(yb)
        train_count += len(yb)

    train_loss = train_loss_sum / train_count

    # ----- Validation -----
    vqc_reupload.eval()
    val_loss_sum = 0.0
    val_count = 0

    with torch.no_grad():
        for xb, yb in valid_loader:
            logits = vqc_reupload(xb)
            loss = criterion(logits, yb)

            val_loss_sum += loss.item() * len(yb)
            val_count += len(yb)

    val_loss = val_loss_sum / val_count

    # ----- Early Stopping -----
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(
            vqc_reupload.state_dict()
        )
        patience_counter = 0
    else:
        patience_counter += 1

    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(
            f"Epoch {epoch + 1:03d} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Valid Loss: {val_loss:.4f}"
        )

    if patience_counter >= PATIENCE:
        print(f"Early stopping at epoch {epoch + 1}")
        break


# 가장 성능이 좋았던 검증 모델 복원
if best_state is not None:
    vqc_reupload.load_state_dict(best_state)

print(f"\nBest validation loss: {best_val_loss:.4f}")


# ==================================================
# 6. Test 평가
# ==================================================

vqc_reupload.eval()

with torch.no_grad():
    test_logits = vqc_reupload(Xte)
    probabilities = torch.sigmoid(test_logits).cpu().numpy()

targets = yte.cpu().numpy().astype(int)
predictions = (probabilities >= 0.5).astype(int)

vqc_reupload_results = {
    "Accuracy": accuracy_score(targets, predictions),
    "Balanced accuracy": balanced_accuracy_score(
        targets, predictions
    ),
    "Precision_code2": precision_score(
        targets, predictions, zero_division=0
    ),
    "Recall_code2": recall_score(
        targets, predictions, zero_division=0
    ),
    "F1_code2": f1_score(
        targets, predictions, zero_division=0
    ),
    "ROC-AUC": roc_auc_score(targets, probabilities),
    "Average precision": average_precision_score(
        targets, probabilities
    ),
    "Log loss": log_loss(
        targets, probabilities, labels=[0, 1]
    ),
}

vqc_reupload_results = pd.Series(
    vqc_reupload_results
)

print("\n[Data Re-uploading VQC 테스트 성능]")
print(vqc_reupload_results.round(4))

print("\n[Confusion Matrix]")
print(confusion_matrix(targets, predictions))

print("\n[Classification Report]")
print(
    classification_report(
        targets,
        predictions,
        digits=4,
        zero_division=0
    )
)


## 모델 성능 비교

In [ ]:
# ============================================================
# 테스트 성능 비교
# ============================================================

comparison = pd.DataFrame({
    "XGBoost(전체 변수)": results.loc["test"],
    "XGBoost(PCA 8개 버전)": results_pca.loc["test"],
    "VQC(8qubit)": vqc_results.loc["test"],
    "VQC(8qubit+drc)": vqc_reupload_results,
}).T

print("\n[테스트 성능 비교]")
print(comparison.round(4).to_string())

## 연구 보고서

2026년 10월 31일

#### 1. 실험 개요

| 항목 | 설정 |
|---|---|
| 데이터 수 | 3,140개 |
| Train / Validation / Test | 1,884 / 628 / 628 |
| 독립변수 | 16개 |
| 타깃 변수 | 정상/폐업 (이진분류) |
| 타깃 매핑 | 원본 코드 1 → 0, 원본 코드 2 → 1 |
| 데이터 분할 | 60% / 20% / 20% |
| 클래스 비율 | 50:50 |
| PCA 차원 | 8개 |
| PCA 누적 설명분산 비율 | 74.62% |
| 양자회로 | 8큐비트 |

**독립변수 16개 구성**

1. 영업이익률
2. 재료비/매출액
3. 감가상각률
4. 금융비용대총부채비율
5. 금융비용대총비용비율
6. EBITDA/총차입금
7. 차입금의존도
8. 장단기대여금/총자산
9. 법인세부담률
10. 순운전자본대총자본비율
11. 유동자산구성비율
12. 유보액대총자산비율
13. 재고자산대유동자산비율
14. 매입채무회전기간
15. 자본금회전율(회)
16. 업력

#### 2. 모델 구성

**공통 전처리**

- 극단값 제한: 훈련셋 기준 1% / 99% 분위수
- 결측값 처리: 중앙값 대치
- 스케일링: RobustScaler
- 검증 및 테스트 데이터는 훈련 데이터에서 학습한 전처리 객체로 transform만 적용
- 최종 입력 데이터: 16개 변수

**A. XGBoost (독립변수 16개 입력)**

| 항목 | 설정 |
|---|---|
| 목적함수 | binary:logistic |
| 트리 수 | 최대 2,000개 |
| 최적 트리 수 | 272개 |
| 조기 종료 | 검증 Log Loss 기준 100회 |
| learning_rate | 0.03 |
| max_depth | 3 |
| min_child_weight | 5 |
| subsample / colsample_bytree | 0.8 / 0.8 |
| reg_alpha / reg_lambda | 0.1 / 5.0 |
| scale_pos_weight | 1.0 |
| tree_method | hist |
| random_state | 42 |

**B. PCA-XGBoost (PCA 8개 입력)**

- 입력: 16개 독립변수
- StandardScaler → PCA(8개 성분)
- PCA는 훈련셋에서만 fit
- 누적 설명분산 비율: 74.62%

| 항목 | 설정 |
|---|---|
| n_estimators | 300 |
| max_depth | 4 |
| learning_rate | 0.05 |
| subsample | 0.8 |
| colsample_bytree | 0.8 |
| objective | binary:logistic |
| eval_metric | logloss |
| random_state | 42 |
| 조기 종료 | 미적용 |

주의: A와 B의 하이퍼파라미터가 동일하지 않음.

**C. VQC (8큐비트)**

| 단계 | 구성 |
|---|---|
| 입력 | 16개 변수 |
| 차원 축소 | VarianceThreshold → StandardScaler → PCA 8개 성분 |
| 성분 표준화 | StandardScaler |
| 각도 변환 | θ = π·tanh(z/2) |
| 인코딩 | AngleEmbedding, RY 회전 |
| 큐비트 수 | 8개 |
| 학습 회로 | StronglyEntanglingLayers 2층 |
| 얽힘 | CNOT 기반 |
| 측정 | 8개 큐비트의 Pauli-Z 기대값 |
| 출력층 | Linear(8 → 1) |
| 학습 파라미터 | 57개 (회로 48 + 출력층 9) |
| 손실함수 | BCEWithLogitsLoss |
| Optimizer | Adam |
| Learning rate | 0.01 |
| Batch size | 64 |
| 최대 Epoch | 100 |
| Early stopping | Patience 15 |
| 최적 Epoch | 69 |
| 최적 검증 Loss | 0.5885 |
| 시뮬레이터 | PennyLane default.qubit |
| 측정 방식 | shots=None |
| 연산 환경 | CPU, float64 |

**D. DR-VQC (8큐비트 + Data Re-uploading)**

| 단계 | 구성 |
|---|---|
| 입력 | PCA 8개 성분 |
| 큐비트 수 | 8개 |
| 데이터 재주입 횟수 | 2회 |
| 인코딩 | AngleEmbedding, RY 회전 |
| 학습 회로 | StronglyEntanglingLayers |
| 재업로딩 블록당 양자층 | 2층 |
| 전체 양자층 | 4층 |
| 측정 | 8개 큐비트의 Pauli-Z 기대값 |
| 출력층 | Linear(8 → 1) |
| 학습 파라미터 | 105개 (회로 96 + 출력층 9) |
| 손실함수 | BCEWithLogitsLoss |
| Optimizer | Adam |
| Learning rate | 0.01 |
| Batch size | 64 |
| 최대 Epoch | 100 |
| Early stopping | Patience 15 |
| 조기 종료 시점 | 34 Epoch |
| 최적 검증 Loss | 0.5621 |
| 시뮬레이터 | PennyLane default.qubit |
| 연산 환경 | CPU, float64 |

#### 3. 모델별 성능 비교

| 지표 | A. XGBoost | B. PCA-XGBoost | C. VQC | D. DR-VQC |
|---|---:|---:|---:|---:|
| Accuracy | **0.8615** | 0.8232 | 0.6990 | 0.7532 |
| Precision | **0.8429** | 0.8182 | 0.7010 | 0.7607 |
| Recall | **0.8885** | 0.8312 | 0.6943 | 0.7389 |
| F1-score | **0.8651** | 0.8246 | 0.6976 | 0.7496 |
| ROC-AUC | **0.9453** | 0.8946 | 0.7791 | 0.8313 |
| AP | **0.9447** | 0.8677 | 0.7863 | 0.8178 |
| Log Loss | **0.3044** | 0.4052 | 0.5619 | 0.5128 |

※ Precision, Recall, F1-score는 원본 타깃 코드 2를 양성 클래스로 계산.

#### 4. 핵심 결과

- 최고 성능: 16개 변수 기반 XGBoost
- PCA 적용 후 XGBoost 성능 하락
- VQC는 PCA-XGBoost보다 낮은 성능
- Data Re-uploading VQC는 기본 VQC보다 성능 향상
- DR-VQC는 기본 VQC보다 학습 파라미터가 57개에서 105개로 증가
- 양자회로의 표현력 개선 가능성을 확인했으나 고전 모델의 성능을 넘지는 못함

#### 5. 전처리 및 변수 선정 과정에서 제외된 변수

| 구분 | 변수 | 처리 방식 |
|---|---|---|
| 식별자 | 회사코드 | 모델 입력에서 제외 |
| 타깃 | 정상/폐업 | 종속변수로 사용 |
| 날짜 | 재무비율기준/결산기준일 | 모델 입력에서 제외 |
| 선택 변수 | 업력(K1) | 기존 실험에서 제외했으나 이번 실험에서는 별도 처리 후 포함 |
| 나머지 재무변수 | 16개 선정 목록 외 변수 | 모델 입력에서 제외 |
| 범주형 변수 | 산업코드, 감사의견코드 | 이번 실험에는 미포함 |

**중요한 차이점**

이번 실험에서는 업력(K1)을 다시 포함했음.

기존 309개 변수 실험에서는 업력이 예측시점 이후 정보를 포함할 가능성이 있어 제외했지만, 이번 실험에서는 16개 변수 선정 과정에서 다시 추가했음.

따라서 업력의 실제 산출 기준과 예측 기준시점을 확인해야 함. 이는 잠재적인 데이터 누수 문제임.

또한 이번 16개 변수는 자동으로 중요도 상위 16개를 추출한 것이 아니라, 코드에 명시된 변수 목록을 직접 선택한 것임.

#### 6. 결론

이번 실험에서는 16개 재무변수만 사용하더라도 XGBoost가 우수한 분류 성능을 보였다.

16개 변수를 직접 입력한 XGBoost의 ROC-AUC는 0.9453이었으며, PCA를 통해 8개 성분으로 축소한 경우 0.8946으로 하락했다.

양자 머신러닝에서는 기본 VQC보다 Data Re-uploading VQC가 더 높은 성능을 기록했다. 그러나 XGBoost 및 PCA-XGBoost의 성능을 넘지는 못했다.

따라서 현재 결과만으로는 양자 머신러닝의 성능 우위를 확인하기 어렵다. 다만 Data Re-uploading을 통한 양자회로의 표현력 개선 가능성은 추가 연구 가치가 있다.

#### 7. 추가 확인 사항

- [ ] 타깃값 원본 코드 1, 2의 실제 의미 확인
- [ ] 업력(K1)의 산출 기준 및 예측시점 이후 정보 포함 여부 확인
- [ ] 업력 포함 / 제외에 따른 성능 비교
- [ ] 랜덤 시드 변경 후 성능 재검증
- [ ] 동일한 PCA 차원에서 XGBoost와 VQC 비교
- [ ] XGBoost의 PCA 버전과 일반 버전의 하이퍼파라미터 통일
- [ ] DR-VQC 재업로딩 횟수에 따른 성능 변화 확인
- [ ] PCA 성분 수 변경에 따른 성능 변화 확인
- [ ] 결산연도에 따른 시점 교란 가능성 확인
- [ ] 시간 순서 기반 Train / Test 분할을 통한 미래 데이터 일반화 성능 검증
- [ ] 동일한 데이터 분할과 여러 랜덤 시드에서 양자회로 성능 안정성 확인

#### 8. 기존 309개 변수 실험과 비교

| 구분 | 309개 변수 실험 | 16개 변수 실험 |
|---|---:|---:|
| XGBoost Accuracy | 0.8933 | 0.8615 |
| XGBoost ROC-AUC | 0.9648 | 0.9453 |
| PCA-XGBoost Accuracy | 0.8280 | 0.8232 |
| VQC Accuracy | 0.8057 | 0.6990 |
| DR-VQC Accuracy | 0.7946 | 0.7532 |

16개 변수로 줄였을 때 XGBoost의 성능 하락보다 VQC의 성능 하락이 더 크게 나타났다.

다만 두 실험의 변수 구성과 전처리 조건이 다르므로, 이를 양자 모델의 변수 축소 취약성으로 단정할 수는 없다. 동일한 조건에서 추가 검증이 필요하다.
권오병 논문에서 양자에 유리한 데이터 추출하는 부분 참고해서 새실험 세팅